# Reconciliation drop-in: regenerate Table I and Table V from the centerpiece harness

T

In [ ]:
# ===================== [PREAMBLE] helpers (skip if reusing your kernel) =====================
import os, io as _io, subprocess, pickle, json
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
import torchvision.models as models
from torchvision.models import ResNet50_Weights
from torchvision.transforms.functional import gaussian_blur
from PIL import Image as _Image
from sklearn.metrics import roc_auc_score

SEED=42; device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
np.random.seed(SEED); torch.manual_seed(SEED)

def _find(name, maxdepth=8):
    roots=['/home','/root','/workspace',os.path.expanduser('~'),'.','..','../..','.']
    res=[]
    for r in roots:
        if not os.path.exists(r): continue
        try:
            out=subprocess.run(['find',r,'-maxdepth',str(maxdepth),'-type','f','-name',name],
                               capture_output=True,text=True,timeout=20).stdout.strip()
            if out: res+=[p for p in out.split('\n') if p]
        except: pass
    return sorted(set(res))

CIFAR_MEAN=[0.4914,0.4822,0.4465]; CIFAR_STD=[0.2470,0.2435,0.2616]
IMGNET_MEAN=[0.485,0.456,0.406]; IMGNET_STD=[0.229,0.224,0.225]
def make_pp(ds):
    m,s=(CIFAR_MEAN,CIFAR_STD) if 'CIFAR' in ds else (IMGNET_MEAN,IMGNET_STD)
    mean=torch.tensor(m).view(1,3,1,1).to(device); std=torch.tensor(s).view(1,3,1,1).to(device)
    return lambda x:(x/255.0-mean)/std
def load_backbone(ds):
    cfg={'CIFAR-10':('resnet50_cifar10_finetuned.pt',10),'CIFAR-100':('resnet50_cifar100_finetuned.pt',100),
         'SVHN':('resnet50_svhn_finetuned.pt',10),'TinyImageNet':('resnet50_tinyimagenet_finetuned.pt',200)}
    if ds in cfg:
        ck=(_find(cfg[ds][0]) or [None])[0]; m=models.resnet50(weights=None); m.fc=nn.Linear(2048,cfg[ds][1])
        m.load_state_dict(torch.load(ck,map_location=device)['state_dict'])
    else:
        m=models.resnet50(weights=ResNet50_Weights.IMAGENET1K_V2)
    return m.to(device).eval()
def gb(x,s):
    k=int(2*np.ceil(3*s)+1); k=k+1 if k%2==0 else k
    return gaussian_blur(x,kernel_size=k,sigma=s)
def to224(img):
    if img.dim()==3: img=img.unsqueeze(0)
    img=img.float()
    if img.shape[-1]!=224: img=F.interpolate(img,size=(224,224),mode='bicubic',align_corners=False)
    return img.clamp(0,255)
def jpeg(x,q=75):
    a=x.detach().squeeze(0).permute(1,2,0).clamp(0,255).byte().cpu().numpy()
    b=_io.BytesIO(); _Image.fromarray(a).save(b,format='JPEG',quality=int(q)); b.seek(0)
    return torch.from_numpy(np.array(_Image.open(b).convert('RGB'))).float().permute(2,0,1)
def jpeg_batch(b,q=75):
    return torch.stack([jpeg(b[i:i+1]) for i in range(b.shape[0])]).to(b.device)
def median3(x):
    p=F.pad(x,(1,1,1,1),mode='reflect')
    return p.unfold(2,3,1).unfold(3,3,1).contiguous().view(*x.shape,9).median(-1).values
def feat_hfe(b):
    return ((b-gb(b,0.5)).abs().flatten(1).mean(1)/255.0).detach().cpu().numpy()
def feat_gl(b,bb,pp,glsig):
    with torch.no_grad():
        p0=F.softmax(bb(pp(b)),1); p1=F.softmax(bb(pp(gb(b,glsig))),1)
    return (p0-p1).abs().sum(1).cpu().numpy()
def feat_predl1(b,bb,pp):
    with torch.no_grad():
        p0=F.softmax(bb(pp(b)),1); sq=median3(jpeg_batch(b)).clamp(0,255); p2=F.softmax(bb(pp(sq)),1)
    return (p0-p2).abs().sum(1).cpu().numpy()
def auc_ci(neg,pos,B=2000,seed=SEED):
    neg=np.asarray(neg);pos=np.asarray(pos)
    base=roc_auc_score(np.r_[np.zeros(len(neg)),np.ones(len(pos))],np.r_[neg,pos])
    rng=np.random.RandomState(seed); b=[]
    for _ in range(B):
        nb=neg[rng.randint(0,len(neg),len(neg))]; pb=pos[rng.randint(0,len(pos),len(pos))]
        b.append(roc_auc_score(np.r_[np.zeros(len(nb)),np.ones(len(pb))],np.r_[nb,pb]))
    return base,float(np.percentile(b,2.5)),float(np.percentile(b,97.5))
def find_mixed():
    out={}
    for p in _find('mixed_dataset.pkl'):
        pl=p.lower()
        if 'cifar10' in pl or 'cifar_10' in pl: out.setdefault('CIFAR-10',p)
        elif 'imagenet' in pl and 'eps8' in pl: out.setdefault('ImageNet',p)
    return out
print('[PREAMBLE] ready')


In [ ]:
# ===================== regenerate Table I and Table V (centerpiece methodology) =====================
N_CLEAN=500; FPRS=[0.01,0.05,0.10]; NAMES=['HF-Energy','GaussianL1','PredL1']

def half(n, seed=SEED):
    rng=np.random.RandomState(seed); idx=np.arange(n); rng.shuffle(idx); return idx[:n//2], idx[n//2:]
def tpr_at_fpr(neg,pos,fpr):
    thr=np.quantile(neg,1-fpr); return float((np.asarray(pos)>=thr).mean())
def feats_all(X, bb, pp, glsig, bs=64):
    H=[];G=[];P=[]
    for i in range(0,len(X),bs):
        b=X[i:i+bs].to(device); H.append(feat_hfe(b)); G.append(feat_gl(b,bb,pp,glsig)); P.append(feat_predl1(b,bb,pp))
    return {'HF-Energy':np.concatenate(H),'GaussianL1':np.concatenate(G),'PredL1':np.concatenate(P)}

MX=find_mixed()
TABLE1={}; TABLE5={}
for ds, pkl in MX.items():
    bb=load_backbone(ds); pp=make_pp(ds); glsig=0.5 if ds=='CIFAR-10' else 1.0
    mixed=pickle.load(open(pkl,'rb'))
    clean=[to224(im).cpu() for (im,lb,atk) in mixed if atk=='clean']
    adv  =[to224(im).cpu() for (im,lb,atk) in mixed if atk!='clean']
    rng=np.random.RandomState(SEED); clean=[clean[i] for i in rng.permutation(len(clean))[:N_CLEAN]]
    Xc=torch.cat(clean,0); Xa=torch.cat(adv,0)
    ci,ti=half(len(clean)); Xcal=Xc[ci]; Xte=Xc[ti]
    # hard negatives on the clean TEST half
    noise=(Xte+torch.randn_like(Xte)*8.0).clamp(0,255)
    jp=jpeg_batch(Xte.to(device)).cpu()
    bl=gb(Xte.to(device),1.0).clamp(0,255).cpu()

    fcal=feats_all(Xcal,bb,pp,glsig); fte=feats_all(Xte,bb,pp,glsig); fa=feats_all(Xa,bb,pp,glsig)
    fn=feats_all(noise,bb,pp,glsig); fj=feats_all(jp,bb,pp,glsig); fb=feats_all(bl,bb,pp,glsig)

    mu={n:fcal[n].mean() for n in NAMES}; sd={n:fcal[n].std()+1e-8 for n in NAMES}
    def Zmat(d): return np.stack([(d[n]-mu[n])/sd[n] for n in NAMES],1)        # [N,3]
    muS=Zmat(fcal).mean(1).mean()
    def get_anom(d):
        out={n:np.abs((d[n]-mu[n])/sd[n]) for n in NAMES}
        out['Mean ens.']=np.abs(Zmat(d).mean(1)-muS); return out
    Ate=get_anom(fte); Aadv=get_anom(fa); An=get_anom(fn); Aj=get_anom(fj); Ab=get_anom(fb)
    feats=NAMES+['Mean ens.']

    # ---- Table I (Protocol A) ----
    T1={}
    for n in feats:
        pos=Aadv[n]; neg_p=Ate[n]
        neg_noise=np.concatenate([Ate[n],An[n]])
        neg_all=np.concatenate([Ate[n],An[n],Aj[n],Ab[n]])
        ap=auc_ci(neg_p,pos); ano=auc_ci(neg_noise,pos); aall=auc_ci(neg_all,pos)
        T1[n]={'pristine':round(ap[0],4),'matched_noise':round(ano[0],4),'all3':round(aall[0],4),
               'delta_auc':round(aall[0]-ap[0],4),
               'ci_pristine':[round(ap[1],4),round(ap[2],4)],'ci_all':[round(aall[1],4),round(aall[2],4)]}
    TABLE1[ds]=T1

    # ---- Table V (operational TPR@FPR) ----
    T5={}
    for n in feats:
        pos=Aadv[n]; neg_p=Ate[n]; neg_hn=np.concatenate([Ate[n],An[n],Aj[n],Ab[n]])
        T5[n]={'P':{f'{int(f*100)}%':round(tpr_at_fpr(neg_p,pos,f),4) for f in FPRS},
               'HN':{f'{int(f*100)}%':round(tpr_at_fpr(neg_hn,pos,f),4) for f in FPRS}}
    TABLE5[ds]=T5
    print(f'[{ds}] clean={len(clean)} adv={Xa.shape[0]}  HF delta-AUC={T1["HF-Energy"]["delta_auc"]}  '
          f'HF opTPR@5%={T5["HF-Energy"]["HN"]["5%"]}')

OUT='./reconcile_results'; os.makedirs(OUT,exist_ok=True)
json.dump({'table1':TABLE1,'table5':TABLE5}, open(os.path.join(OUT,'table1_table5.json'),'w'), indent=2)
print('saved', os.path.join(OUT,'table1_table5.json'))

import pandas as pd
print('\n=== Table I (per dataset) ===')
for ds in TABLE1:
    print(ds)
    print(pd.DataFrame(TABLE1[ds]).T[['pristine','matched_noise','all3','delta_auc']])
print('\n=== Table V (TPR@FPR) ===')
for ds in TABLE5:
    print(ds)
    print(pd.DataFrame({n:{'@5%(P)':TABLE5[ds][n]['P']['5%'],'@5%(HN)':TABLE5[ds][n]['HN']['5%']} for n in TABLE5[ds]}).T)


## What to upload back
Upload the executed notebook and `reconcile_results/table1_table5.json`. I will overwrite Table I and
Table V in the manuscript with these values so all three tables (TABLE 0, Table I, Table V) share one
methodology, and adjust the surrounding prose (e.g. the operational-TPR wording, which becomes a
noise-dominated collapse to about 0.33 on CIFAR-10 rather than the previous 0.000).

**Sanity checks while it runs**
- HF-Energy CIFAR-10 should print delta-AUC about -0.17 and operational TPR@5% about 0.33, matching
  TABLE 0 (not the old -0.306 / 0.000).
- "Mean ens." should track HF-Energy closely on CIFAR-10 (the mean is dragged by HF's large z), which
  is the mechanism Sec. V-C describes.
- Table I "pristine" for HF-Energy should be about 1.000 on CIFAR-10 and about 0.55 on ImageNet.
